# **Feature Extraction** for NeoScreen
## Extracting audio features from all three datasets: **ICBHI 2017**, **HLS-CMDS**, **SPRSound**

## 1. Imports

In [1]:

import os
import sys
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from pathlib import Path
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# Add project root to path
sys.path.append('..')

from EDA.audio_features import AudioFeatureExtractor

print("Imports successful")

Imports successful


## 2. Define path

In [7]:
sprsound_path = Path("../../sound_data/sprsound")
output_dir = Path("../../sound_data/features")
output_dir.mkdir(parents=True, exist_ok=True)

def find_sprsound_wavs(base_path):
    wav_files = []
    # os.walk handles nested year folders (2022, 2023, etc.)
    for root, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith('.wav'):
                wav_files.append(Path(root) / file)
    return wav_files

## 3. Load Labels from EDA

In [3]:
# 1. Load the clean labels generated in the EDA
df_labels = pd.read_csv(sprsound_path / "sprsound_file_map_clean.csv")

# Create a dictionary mapping filename to actual symptom
label_dict = dict(zip(df_labels['filename'], df_labels['actual_symptom']))

# 2. Find all original files
if sprsound_path.exists():
    original_wavs = find_sprsound_wavs(sprsound_path)
    print(f"Found {len(original_wavs)} original SPRSound files.")
else:
    print(f"SPRSound folder not found at: {sprsound_path.absolute()}")

Found 10097 original SPRSound files.


## 4. Seup Augmentation

In [4]:
# Setup Augmentation Directory for the new physical .wav files
aug_dir = Path("../../sound_data/sprsound/augmented_wavs")
aug_dir.mkdir(parents=True, exist_ok=True)

# Classes we want to multiply by 6 (1 original + 5 augmented)
minority_classes = ['stridor', 'rhonchi', 'wheeze_crackle', 'wheeze']
all_processing_files = []
expanded_labels = []

print("\nApplying Acoustic Augmentations to Minority Classes...")
for wav_path in tqdm(original_wavs):
    filename = wav_path.name
    symptom = label_dict.get(filename, 'other')
    
    # Always keep the original file!
    all_processing_files.append(wav_path)
    expanded_labels.append({'filename': filename, 'symptom': symptom, 'is_augmented': 0})
    
    # ONLY augment if it's a minority distress sound
    if symptom in minority_classes:
        try:
            # Load raw audio
            y, sr = librosa.load(wav_path, sr=None)
            base_name = wav_path.stem
            
            # 1. White Noise
            noise_amp = 0.005 * np.random.uniform() * np.amax(y)
            y_noise = y + noise_amp * np.random.normal(size=y.shape[0])
            p1 = aug_dir / f"{base_name}_aug_noise.wav"
            sf.write(p1, y_noise, sr)
            all_processing_files.append(p1)
            expanded_labels.append({'filename': p1.name, 'symptom': symptom, 'is_augmented': 1})
            
            # 2. Pitch Shift Up (+1.5 half steps)
            y_pitch_up = librosa.effects.pitch_shift(y, sr=sr, n_steps=1.5)
            p2 = aug_dir / f"{base_name}_aug_pitchup.wav"
            sf.write(p2, y_pitch_up, sr)
            all_processing_files.append(p2)
            expanded_labels.append({'filename': p2.name, 'symptom': symptom, 'is_augmented': 1})
            
            # 3. Pitch Shift Down (-1.5 half steps)
            y_pitch_down = librosa.effects.pitch_shift(y, sr=sr, n_steps=-1.5)
            p3 = aug_dir / f"{base_name}_aug_pitchdown.wav"
            sf.write(p3, y_pitch_down, sr)
            all_processing_files.append(p3)
            expanded_labels.append({'filename': p3.name, 'symptom': symptom, 'is_augmented': 1})
            
            # 4. Time Stretch Fast (10% faster)
            y_stretch_fast = librosa.effects.time_stretch(y, rate=1.1)
            p4 = aug_dir / f"{base_name}_aug_fast.wav"
            sf.write(p4, y_stretch_fast, sr)
            all_processing_files.append(p4)
            expanded_labels.append({'filename': p4.name, 'symptom': symptom, 'is_augmented': 1})
            
            # 5. Time Stretch Slow (10% slower)
            y_stretch_slow = librosa.effects.time_stretch(y, rate=0.9)
            p5 = aug_dir / f"{base_name}_aug_slow.wav"
            sf.write(p5, y_stretch_slow, sr)
            all_processing_files.append(p5)
            expanded_labels.append({'filename': p5.name, 'symptom': symptom, 'is_augmented': 1})
            
        except Exception as e:
            # Skip file if librosa fails to process it
            continue

# Save the master label map so we can use it in the final preparation notebook
df_expanded_labels = pd.DataFrame(expanded_labels)
df_expanded_labels.to_csv(output_dir / "sprsound_labels_master.csv", index=False)
print(f"\nAugmentation Complete! Total files to process: {len(all_processing_files)}")


Applying Acoustic Augmentations to Minority Classes...


  0%|          | 0/10097 [00:00<?, ?it/s]


Augmentation Complete! Total files to process: 13627


## 5. Initialize Feature Extractor

In [5]:
# Run the Extractor on EVERYTHING
print("\nStarting batch extraction on all original and augmented files...")

extractor = AudioFeatureExtractor()
df_sprsound = extractor.extract_batch(
    all_processing_files, 
    output_csv=output_dir / "sprsound_augmented_features.csv"
)


Starting batch extraction on all original and augmented files...
AudioFeatureExtractor initialized with sr=4000, n_mfcc=13

Processing 13627 files...
   Progress: 0/13627 (Success: 0, Failed: 0)
   Progress: 10/13627 (Success: 10, Failed: 0)
   Progress: 20/13627 (Success: 20, Failed: 0)
   Progress: 30/13627 (Success: 30, Failed: 0)
   Progress: 40/13627 (Success: 40, Failed: 0)
   Progress: 50/13627 (Success: 50, Failed: 0)
   Progress: 60/13627 (Success: 60, Failed: 0)
   Progress: 70/13627 (Success: 70, Failed: 0)
   Progress: 80/13627 (Success: 80, Failed: 0)
   Progress: 90/13627 (Success: 90, Failed: 0)
   Progress: 100/13627 (Success: 100, Failed: 0)
   Progress: 110/13627 (Success: 110, Failed: 0)
   Progress: 120/13627 (Success: 120, Failed: 0)
   Progress: 130/13627 (Success: 130, Failed: 0)
   Progress: 140/13627 (Success: 140, Failed: 0)
   Progress: 150/13627 (Success: 150, Failed: 0)
   Progress: 160/13627 (Success: 160, Failed: 0)
   Progress: 170/13627 (Success: 170, 

## 6. Merge Labels

In [8]:
# Merge the True Labels onto the extracted features!
df_final_features = pd.merge(df_sprsound, df_expanded_labels, on='filename', how='left')
df_final_features.to_csv(output_dir / "sprsound_augmented_features.csv", index=False)

print(f"\nExtraction complete! Final Feature Matrix Shape: {df_final_features.shape}")
print("\nNew Class Distribution ready for modeling:")
print(df_final_features['symptom'].value_counts())


Extraction complete! Final Feature Matrix Shape: (20687, 37)

New Class Distribution ready for modeling:
symptom
other             11897
wheeze             4191
wheeze_crackle     2431
crackles           1024
rhonchi             825
stridor             319
Name: count, dtype: int64
